In [34]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id":    [1,2,3,4,5,6,7,8,9,10],
    "name":           ["Alice","Bob","Charlie","Diana","Eve",
                       "Frank","Grace","Henry","Isla","Jack"],
    "region":         ["North","South","North","East","South",
                       "East","North","South","East","North"],
    "age":            [25,34,45,23,52,31,40,29,38,47],
    "join_date":      ["2022-01-15","2021-06-20","2020-03-10","2023-02-28",
                       "2019-11-05","2022-08-14","2021-03-22","2023-05-10",
                       "2020-09-18","2019-07-30"]
})

transactions = pd.DataFrame({
    "transaction_id": list(range(1,21)),
    "customer_id":    [1,2,1,3,4,2,5,6,3,7,8,4,9,10,5,6,7,8,9,10],
    "amount":         [120,340,85,210,95,420,180,310,140,260,
                       390,175,225,480,90,350,270,415,195,310],
    "category":       ["Electronics","Clothing","Food","Electronics","Food",
                       "Clothing","Electronics","Clothing","Food","Electronics",
                       "Clothing","Food","Electronics","Clothing","Food",
                       "Electronics","Clothing","Food","Electronics","Clothing"],
    "date":           ["2024-01-05","2024-01-12","2024-01-20","2024-02-03",
                       "2024-02-14","2024-02-25","2024-03-05","2024-03-12",
                       "2024-03-18","2024-03-25","2024-04-02","2024-04-10",
                       "2024-04-15","2024-04-20","2024-05-01","2024-05-10",
                       "2024-05-15","2024-05-20","2024-06-01","2024-06-10"]
})

def segment_customers(customers, transactions):

    # Step 1: Convert join_date and date columns to datetime
    customers["join_date"] = pd.to_datetime(customers["join_date"], format="%Y-%m-%d")
    transactions["date"] = pd.to_datetime(transactions["date"], format="%Y-%m-%d")
    

    # Step 2: Calculate total spend per customer from transactions
    total_spend = transactions.groupby("customer_id")["amount"].sum()


    # Step 3: Calculate number of transactions per customer
    transaction_count = transactions.groupby("customer_id")["transaction_id"].count()


    # Step 4: Merge customers with total spend and transaction count
    merge = pd.merge(customers, total_spend, on = "customer_id", how = "left")
    merged = pd.merge(merge, transaction_count, on = "customer_id", how = "left")
    merged = merged.rename(columns={
        "amount": "total_spend",
        "transaction_id": "transaction_count"
    })

    # Step 5: Add a column 'segment':
    # 'High Value'   — total spend > 500
    # 'Mid Value'    — total spend between 200 and 500
    # 'Low Value'    — total spend <= 200
    conditions = [
        merged["total_spend"] > 500,
        (merged["total_spend"] > 200) & (merged["total_spend"] <=500),
        merged["total_spend"]<=200
    ]
    choices = ["High Value", "Mid Value", "Low Value"]
    merged["segment"] = np.select(conditions, choices, default="Low Value")


    # Step 6: Count customers per segment
    num_of_customers = merged.groupby("segment")["customer_id"].count()
    
    # Step 7: Average spend per region
    average_spend_per_region = merged.groupby("region")["total_spend"].mean()
    
    # Step 8: Return summary dict with keys:
    # 'segment_counts', 'avg_spend_by_region', 'top_customer'
    # top_customer = name of customer with highest total spend
    average_spend_per_name = merged.groupby("name")["total_spend"].mean()

    summary= {
        "segment_counts": num_of_customers,
        "avg_spend_by_region" : average_spend_per_region,
        "top_customer":average_spend_per_name.idxmax()
    }
    
    return summary

print(segment_customers(customers, transactions))

{'segment_counts': segment
High Value    5
Mid Value     5
Name: customer_id, dtype: int64, 'avg_spend_by_region': region
East     450.000000
North    468.750000
South    611.666667
Name: total_spend, dtype: float64, 'top_customer': 'Henry'}


In [4]:
# Step 1: Convert join_date and date columns to datetime
customers["join_date"] = pd.to_datetime(customers["join_date"], format="%Y-%m-%d")
transactions["date"] = pd.to_datetime(transactions["date"], format="%Y-%m-%d")

In [5]:
transactions.groupby("customer_id")["amount"].sum()

customer_id
1     205
2     760
3     350
4     270
5     270
6     660
7     530
8     805
9     420
10    790
Name: amount, dtype: int64

In [6]:
transactions.groupby("customer_id")["transaction_id"].count()

customer_id
1     2
2     2
3     2
4     2
5     2
6     2
7     2
8     2
9     2
10    2
Name: transaction_id, dtype: int64

In [16]:
# Step 2: Calculate total spend per customer from transactions
total_spend = transactions.groupby("customer_id")["amount"].sum()

    # Step 3: Calculate number of transactions per customer
transaction_count = transactions.groupby("customer_id")["transaction_id"].count()


merge = pd.merge(customers, total_spend, on = "customer_id", how = "left")
merged = pd.merge(merge, transaction_count, on = "customer_id", how = "left")
merged = merged.rename(columns={
    "amount": "total_spend",
    "transaction_id": "transaction_count"
})
merged.head()

,customer_id,name,region,age,join_date,total_spend,transaction_count
0,1,Alice,North,25,2022-01-15,205,2
1,2,Bob,South,34,2021-06-20,760,2
2,3,Charlie,North,45,2020-03-10,350,2
3,4,Diana,East,23,2023-02-28,270,2
4,5,Eve,South,52,2019-11-05,270,2


In [ ]:
# Step 5: Add a column 'segment':
# 'High Value'   — total spend > 500
# 'Mid Value'    — total spend between 200 and 500
# 'Low Value'    — total spend <= 200

import numpy as np

conditions = [
    merged["total_spend"] > 500,
    (merged["total_spend"] > 200) & (merged["total_spend"] <=500),
    merged["total_spend"]<=200
]
choices = ["High Value", "Mid Value", "Low Value"]

merged["segment"] = np.select(conditions, choices, default="Low Value")

merged.head()


"""
merged["segment"] = pd.cut(
    merged["total_spend"],
    bins=[0, 200, 500, float("inf")],
    labels=["Low Value", "Mid Value", "High Value"]
)
"""
    

,customer_id,name,region,age,join_date,total_spend,transaction_count,segment
0,1,Alice,North,25,2022-01-15,205,2,Mid Value
1,2,Bob,South,34,2021-06-20,760,2,High Value
2,3,Charlie,North,45,2020-03-10,350,2,Mid Value
3,4,Diana,East,23,2023-02-28,270,2,Mid Value
4,5,Eve,South,52,2019-11-05,270,2,Mid Value


In [26]:
num_of_customers = merged.groupby("segment")["customer_id"].count()
num_of_customers

segment
High Value    5
Mid Value     5
Name: customer_id, dtype: int64

In [ ]:
average_spend_per_region = merged.groupby("region")["total_spend"].mean()
average_spend_per_region

region
East     450.000000
North    468.750000
South    611.666667
Name: total_spend, dtype: float64

In [31]:
average_spend_per_name = merged.groupby("name")["total_spend"].mean()
average_spend_per_name

name
Alice      205.0
Bob        760.0
Charlie    350.0
Diana      270.0
Eve        270.0
Frank      660.0
Grace      530.0
Henry      805.0
Isla       420.0
Jack       790.0
Name: total_spend, dtype: float64